In [ ]:
pip install transformers datasets peft accelerate evaluate

In [ ]:
pip install --upgrade transformers

In [ ]:
import os
import random
from typing import Dict, Any, Callable, List
import math
import time
import json

import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



#  1. Data loading & preprocessing

def load_emotion_dataset(
    model_name: str = "distilbert-base-uncased",
    max_length: int = 128,
    train_subset: int = 3000,
    batch_size: int = 32,
):

    dataset = load_dataset("dair-ai/emotion")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = batch["label"]
        return enc

    encoded = dataset.map(preprocess, batched=True)

    if train_subset is not None and train_subset < len(encoded["train"]):
        encoded["train"] = encoded["train"].shuffle(seed=42).select(range(train_subset))

    encoded.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )

    train_loader = DataLoader(encoded["train"], batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(encoded["validation"], batch_size=batch_size)
    test_loader = DataLoader(encoded["test"], batch_size=batch_size)

    num_labels = dataset["train"].features["label"].num_classes
    id2label = {i: dataset["train"].features["label"].int2str(i) for i in range(num_labels)}
    label2id = {v: k for k, v in id2label.items()}

    return train_loader, val_loader, test_loader, tokenizer, num_labels, id2label, label2id



#  2. LoRA model construction and training

def build_lora_model(
    base_model_name: str,
    num_labels: int,
    cfg: Dict[str, Any],
    id2label: Dict[int, str],
    label2id: Dict[str, int],
):
    """
    Build a DistilBERT + LoRA classification model according to cfg:
        cfg["lora_r"], cfg["lora_alpha"], cfg["lora_dropout"], cfg["target_modules"]
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    # Freeze base model to train only LoRA + classifier
    for param in model.base_model.parameters():
        param.requires_grad = False

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["target_modules"],
    )

    model = get_peft_model(model, lora_config)
    return model


def train_one_model(
    cfg: Dict[str, Any],
    train_loader: DataLoader,
    val_loader: DataLoader,
    base_model_name: str,
    num_labels: int,
    id2label: Dict[int, str],
    label2id: Dict[str, int],
    device: torch.device,
    num_epochs: int = 3,
) -> float:
    """
    Train DistilBERT + LoRA with hyperparameters in cfg.
    Returns validation accuracy (float in [0,1]).
    """
    model = build_lora_model(
        base_model_name=base_model_name,
        num_labels=num_labels,
        cfg=cfg,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=cfg["learning_rate"])

    # Total training steps
    total_steps = num_epochs * len(train_loader)
    warmup_steps = int(cfg["warmup_ratio"] * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"    Epoch {epoch+1}/{num_epochs} - train loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            labels = batch["labels"]
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total if total > 0 else 0.0

    # Clean up
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return val_acc


In [ ]:
#  3. Hyperparameter decoding (LoRA search space)

def decode_candidate(x: np.ndarray) -> Dict[str, Any]:
    rank_choices = [2, 4, 8, 16, 32]
    alpha_choices = [8, 16, 32, 64, 128]

    def pick_choice(val: float, choices):
        idx = int(round(val * (len(choices) - 1)))
        idx = max(0, min(len(choices) - 1, idx))
        return choices[idx]


    lr_min = 1e-6
    lr_max = 5e-4
    log_min = math.log(lr_min)
    log_max = math.log(lr_max)
    log_lr = log_min + float(x[0]) * (log_max - log_min)
    learning_rate = math.exp(log_lr)


    warmup_ratio = 0.2 * float(x[1])

    lora_r = pick_choice(x[2], rank_choices)
    lora_alpha = pick_choice(x[3], alpha_choices)


    lora_dropout = 0.3 * float(x[4])


    if x[5] < 0.5:
        target_modules = ["q_lin", "v_lin"]  # attention-only
        target_modules_option = "attention-only"
    else:
        target_modules = ["q_lin", "v_lin", "ffn.lin1", "ffn.lin2"]  # attention+ffn
        target_modules_option = "attention-plus-ffn"

    cfg = {
        "learning_rate": learning_rate,
        "warmup_ratio": warmup_ratio,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "target_modules": target_modules,
        "target_modules_option": target_modules_option,
    }
    return cfg



#  4. Fitness wrapper

def make_fitness_function(
    train_loader: DataLoader,
    val_loader: DataLoader,
    base_model_name: str,
    num_labels: int,
    id2label: Dict[int, str],
    label2id: Dict[str, int],
    device: torch.device,
    num_epochs: int,
) -> Callable[[Dict[str, Any]], float]:


    def fitness(cfg: Dict[str, Any]) -> float:
        print("  Evaluating config:", cfg)
        val_acc = train_one_model(
            cfg=cfg,
            train_loader=train_loader,
            val_loader=val_loader,
            base_model_name=base_model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
            device=device,
            num_epochs=num_epochs,
        )
        print(f"  -> val_acc (fitness) = {val_acc:.4f}")
        return val_acc

    return fitness

In [ ]:
#  5. Random Search for a single seed

def random_search_for_seed(
    num_trials: int,
    fitness_function: Callable[[Dict[str, Any]], float],
    seed: int,
    device: torch.device,
) -> Dict[str, Any]:

    dim = 6
    global_best_val_acc = float("-inf")
    global_best_trial_no = 0
    global_best_trial_index = -1

    trials: List[Dict[str, Any]] = []
    val_accs: List[float] = []
    train_times: List[float] = []
    gpu_mems: List[float] = []

    print(f"\n==============================")
    print(f" Random Search for seed {seed} ")
    print(f"==============================")

    for t in range(1, num_trials + 1):
        print(f"\n[Random Search] Seed {seed} - Trial {t}/{num_trials}")
        x = np.random.rand(dim)
        cfg = decode_candidate(x)

        # GPU memory tracking
        if device.type == "cuda" and torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        start_t = time.perf_counter()
        val_acc = fitness_function(cfg)
        end_t = time.perf_counter()
        train_time = end_t - start_t

        if device.type == "cuda" and torch.cuda.is_available():
            max_mem_bytes = torch.cuda.max_memory_allocated()
            gpu_memory_gb = max_mem_bytes / (1024 ** 3)
        else:
            gpu_memory_gb = None

        val_accs.append(val_acc)
        train_times.append(train_time)
        if gpu_memory_gb is not None:
            gpu_mems.append(gpu_memory_gb)

        if val_acc > global_best_val_acc:
            global_best_val_acc = val_acc
            global_best_trial_no = t
            global_best_trial_index = t - 1

        trial_entry = {
            "trial_no": t,
            "gpu_memory_gb": float(gpu_memory_gb) if gpu_memory_gb is not None else None,
            "val_acc": float(val_acc),
            "best_val_acc_so_far": float(global_best_val_acc),
            "train_time_sec": float(train_time),
            "hyperparams": {
                "learning_rate": float(cfg["learning_rate"]),
                "warmup_ratio": float(cfg["warmup_ratio"]),
                "lora_r": int(cfg["lora_r"]),
                "lora_alpha": int(cfg["lora_alpha"]),
                "lora_dropout": float(cfg["lora_dropout"]),
                "target_modules_option": cfg["target_modules_option"],
            },
        }

        trials.append(trial_entry)

        print(
            f"[Random Search] Seed {seed} | Trial {t} | "
            f"val_acc={val_acc:.4f} | best_val_acc_so_far={global_best_val_acc:.4f}"
        )

    if val_accs:
        val_acc_arr = np.array(val_accs, dtype=float)
        train_time_arr = np.array(train_times, dtype=float)
        val_acc_mean = float(val_acc_arr.mean())
        val_acc_std = float(val_acc_arr.std(ddof=0))
        total_train_time = float(train_time_arr.sum())
        avg_train_time = float(train_time_arr.mean())
    else:
        val_acc_mean = 0.0
        val_acc_std = 0.0
        total_train_time = 0.0
        avg_train_time = 0.0

    if gpu_mems:
        gpu_arr = np.array(gpu_mems, dtype=float)
        avg_gpu_memory_gb = float(gpu_arr.mean())
    else:
        avg_gpu_memory_gb = None

    print(f"\n[Random Search] Seed {seed} - avg GPU memory (GB) over {num_trials} trials: {avg_gpu_memory_gb}")

    seed_result = {
        "seed": int(seed),
        "trials": trials,
        "best_trial_index": int(global_best_trial_index),
        "best_trial_no": int(global_best_trial_no),
        "best_val_acc": float(global_best_val_acc) if global_best_val_acc != float("-inf") else 0.0,
        "val_acc_mean": val_acc_mean,
        "val_acc_std": val_acc_std,
        "avg_train_time_sec": avg_train_time,
        "total_train_time_sec": total_train_time,
        "avg_gpu_memory_gb": avg_gpu_memory_gb,   # <<< NEW: per-seed average GPU usage
    }

    return seed_result

In [ ]:
#  6. run RS with 4 seeds

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base_model_name = "distilbert-base-uncased"


    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
    else:
        gpu_name = "CPU"

    NUM_EPOCHS_PER_TRIAL = 3
    NUM_TRIALS_PER_SEED = 20
    SEEDS = [1, 42, 999, 1234]

    print("Loading Emotion dataset...")
    train_loader, val_loader, test_loader, tokenizer, num_labels, id2label, label2id = load_emotion_dataset(
        model_name=base_model_name,
        max_length=128,
        train_subset=3000,
        batch_size=32,
    )

    print("Building fitness function for Random Search...")
    fitness_fn = make_fitness_function(
        train_loader=train_loader,
        val_loader=val_loader,
        base_model_name=base_model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        device=device,
        num_epochs=NUM_EPOCHS_PER_TRIAL,
    )

    # Search space metadata
    search_space = {
        "learning_rate": {
            "type": "log_uniform",
            "min": 1e-6,
            "max": 5e-4,
        },
        "warmup_ratio": {
            "type": "continuous",
            "min": 0.0,
            "max": 0.2,
        },
        "lora_r": {
            "type": "discrete",
            "values": [2, 4, 8, 16, 32],
        },
        "lora_alpha": {
            "type": "discrete",
            "values": [8, 16, 32, 64, 128],
        },
        "lora_dropout": {
            "type": "continuous",
            "min": 0.0,
            "max": 0.3,
        },
        "target_modules": {
            "type": "categorical",
            "values": [
                "attention-only",
                "attention-plus-ffn",
            ],
        },
    }

    seeds_results: List[Dict[str, Any]] = []

    for seed in SEEDS:
        set_seed(seed)
        seed_result = random_search_for_seed(
            num_trials=NUM_TRIALS_PER_SEED,
            fitness_function=fitness_fn,
            seed=seed,
            device=device,
        )
        seeds_results.append(seed_result)

    # Compute overall_stats
    all_val_accs: List[float] = []
    seed_best_vals: List[float] = []
    total_train_time_all_seeds: float = 0.0

    global_best_val_acc = float("-inf")
    global_best_seed = None
    global_best_trial_no = None
    global_best_hyperparams = None

    for seed_res in seeds_results:
        seed = seed_res["seed"]
        best_val = seed_res["best_val_acc"]
        best_idx = seed_res["best_trial_index"]

        seed_best_vals.append(best_val)
        total_train_time_all_seeds += seed_res["total_train_time_sec"]

        for trial in seed_res["trials"]:
            all_val_accs.append(trial["val_acc"])

        if best_val > global_best_val_acc and best_idx >= 0:
            global_best_val_acc = best_val
            global_best_seed = seed
            global_best_trial_no = seed_res["best_trial_no"]
            global_best_hyperparams = seed_res["trials"][best_idx]["hyperparams"]

    if all_val_accs:
        arr_all = np.array(all_val_accs, dtype=float)
        val_acc_mean_over_all_trials = float(arr_all.mean())
        val_acc_std_over_all_trials = float(arr_all.std(ddof=0))
    else:
        val_acc_mean_over_all_trials = 0.0
        val_acc_std_over_all_trials = 0.0

    if seed_best_vals:
        arr_seed_best = np.array(seed_best_vals, dtype=float)
        val_acc_mean_of_seed_bests = float(arr_seed_best.mean())
        val_acc_std_of_seed_bests = float(arr_seed_best.std(ddof=0))
    else:
        val_acc_mean_of_seed_bests = 0.0
        val_acc_std_of_seed_bests = 0.0

    overall_stats = {
        "global_best_seed": int(global_best_seed) if global_best_seed is not None else None,
        "global_best_trial_no": int(global_best_trial_no) if global_best_trial_no is not None else None,
        "global_best_val_acc": float(global_best_val_acc) if global_best_val_acc != float("-inf") else 0.0,
        "global_best_hyperparams": global_best_hyperparams,
        "val_acc_mean_over_all_trials": val_acc_mean_over_all_trials,
        "val_acc_std_over_all_trials": val_acc_std_over_all_trials,
        "val_acc_mean_of_seed_bests": val_acc_mean_of_seed_bests,
        "val_acc_std_of_seed_bests": val_acc_std_of_seed_bests,
        "total_train_time_sec_over_all_seeds": float(total_train_time_all_seeds),
    }

    # Final JSON
    results = {
        "method": "random_search",
        "gpu_name": gpu_name,
        "num_seeds": len(SEEDS),
        "seeds": SEEDS,
        "total_trials_per_seed": NUM_TRIALS_PER_SEED,
        "num_epochs_per_trial": NUM_EPOCHS_PER_TRIAL,
        "search_space": search_space,
        "seeds_results": seeds_results,
        "overall_stats": overall_stats,
    }

    print("\n==============================")
    print(" RANDOM SEARCH JSON RESULTS ")
    print("==============================")
    print(json.dumps(results, indent=2))

    out_path = "random_search_results.json"
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved Random Search results JSON to: {out_path}")


if __name__ == "__main__":
    main()


Loading Emotion dataset...


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

  -> val_acc (fitness) = 0.5625
[Random Search] Seed 1234 | Trial 17 | val_acc=0.5625 | best_val_acc_so_far=0.8940

[Random Search] Seed 1234 - Trial 18/20
  Evaluating config: {'learning_rate': 0.00014148833186220696, 'warmup_ratio': 0.1115521656854899, 'lora_r': 32, 'lora_alpha': 16, 'lora_dropout': 0.008894100160624674, 'target_modules': ['q_lin', 'v_lin', 'ffn.lin1', 'ffn.lin2'], 'target_modules_option': 'attention-plus-ffn'}
    Epoch 1/3 - train loss: 1.4139
    Epoch 2/3 - train loss: 0.8434
    Epoch 3/3 - train loss: 0.6370


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  -> val_acc (fitness) = 0.7650
[Random Search] Seed 1234 | Trial 18 | val_acc=0.7650 | best_val_acc_so_far=0.8940

[Random Search] Seed 1234 - Trial 19/20
  Evaluating config: {'learning_rate': 2.031701491546504e-06, 'warmup_ratio': 0.19016197001682444, 'lora_r': 4, 'lora_alpha': 16, 'lora_dropout': 0.13734349466922827, 'target_modules': ['q_lin', 'v_lin', 'ffn.lin1', 'ffn.lin2'], 'target_modules_option': 'attention-plus-ffn'}
    Epoch 1/3 - train loss: 1.7721
    Epoch 2/3 - train loss: 1.7404
    Epoch 3/3 - train loss: 1.7276


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  -> val_acc (fitness) = 0.3610
[Random Search] Seed 1234 | Trial 19 | val_acc=0.3610 | best_val_acc_so_far=0.8940

[Random Search] Seed 1234 - Trial 20/20
  Evaluating config: {'learning_rate': 0.00023582036035063445, 'warmup_ratio': 0.050523151009306044, 'lora_r': 4, 'lora_alpha': 16, 'lora_dropout': 0.2705388154112976, 'target_modules': ['q_lin', 'v_lin', 'ffn.lin1', 'ffn.lin2'], 'target_modules_option': 'attention-plus-ffn'}
    Epoch 1/3 - train loss: 1.3223
    Epoch 2/3 - train loss: 0.7211
    Epoch 3/3 - train loss: 0.5284
  -> val_acc (fitness) = 0.8020
[Random Search] Seed 1234 | Trial 20 | val_acc=0.8020 | best_val_acc_so_far=0.8940

[Random Search] Seed 1234 - avg GPU memory (GB) over 20 trials: 1.834447193145752

 RANDOM SEARCH JSON RESULTS 
{
  "method": "random_search",
  "gpu_name": "Tesla T4",
  "num_seeds": 4,
  "seeds": [
    1,
    42,
    999,
    1234
  ],
  "total_trials_per_seed": 20,
  "num_epochs_per_trial": 3,
  "search_space": {
    "learning_rate": {
     